In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve

url = "https://raw.githubusercontent.com/user257814938/PSTB-DI-Bootcamp/refs/heads/main/Week4/Statistics_for_Machine_Learning/DailyChallenge/Churn_Modelling.csv"
df = pd.read_csv(url)

print(df.shape)
print(df.head())
print(df.info())
print(df.describe())

df = df.drop(columns=["RowNumber","CustomerId","Surname"])
print(df.isnull().sum())

le_geo = LabelEncoder()
le_gen = LabelEncoder()
df["Geography"] = le_geo.fit_transform(df["Geography"])
df["Gender"] = le_gen.fit_transform(df["Gender"])

plt.figure(figsize=(6,4))
sns.countplot(x="Exited", data=df)
plt.title("Churn Distribution")
plt.show()

corr = df.corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

X = df.drop("Exited", axis=1)
y = df["Exited"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

acc = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
print("Accuracy:", acc)
print("ROC-AUC:", roc_auc)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f"ROC Curve (AUC={roc_auc:.2f})")
plt.plot([0,1],[0,1],'--',color='gray')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Churn Prediction")
plt.legend()
plt.show()

feature_importance = pd.DataFrame({"Feature": df.drop("Exited", axis=1).columns, "Coefficient": model.coef_[0]})
feature_importance = feature_importance.sort_values(by="Coefficient", ascending=False)
plt.figure(figsize=(8,5))
sns.barplot(x="Coefficient", y="Feature", data=feature_importance)
plt.title("Feature Importance (Logistic Regression Coefficients)")
plt.show()

print("\nBusiness Insights:")
print("1. Customers with high balance and fewer products tend to churn more frequently.")
print("2. Older customers with longer tenure show more loyalty.")
print("3. Geographic and gender differences slightly influence churn.")
print("4. The model can help focus retention strategies on high-risk profiles based on balance, tenure, and activity.")
